# Semantic Search using PineCone DB (Cloud Based)

website: https://www.pinecone.io/

## Objective

1) Convert text → embeddings

2) Store in vector database

3) Perform semantic search

In [1]:
# Following too long time
!pip install pinecone 
# sentence-transformers

# SO I used following on  command line: Took 15 minutes
# python -m pip install pinecone sentence-transformers

  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
   ---------------------------------------- 0.0/742.7 kB ? eta -:--:--
   ---------------------------------------- 742.7/742.7 kB 7.6 MB/s  0:00:00
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)

   ---------------------------------------- 0/5 [pinecone-plugin-interface]
  Attempting uninstall: packaging
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Found existing installation: packaging 25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Uninstalling packaging-25.0:
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
      Successfully uninstalled packaging-25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
   -------- ---------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import Librarairs: Took a 4-5 minutes
from pinecone import Pinecone, ServerlessSpec
# from sentence_transformers import SentenceTransformer

In [5]:
from importlib import metadata
from pinecone import Pinecone, ServerlessSpec

# Get version via package metadata
version = metadata.version("pinecone")
print(version)

7.3.0


In [3]:
# Need API Key
pc = Pinecone(api_key="pcsk_TsJRb_FhL8QjwDpPVsC3CE8h4w5J8uNARnCqgtheDjnEewAbihUAtuWNmKfKB92jZVmQH")
# pc = Pinecone(api_key="pcsk_KEY")

print("🔄 Testing Pinecone API Key connectivity...")

try:
    # 2. Trigger a control plane API call to verify the key
    active_indexes = pc.list_indexes()
    
    print("✅ Success! Your Pinecone API key is valid and authenticated.")
    print(f"📁 Available Indexes in your project: {len(active_indexes)}")
    for index in active_indexes:
        print(f"   - {index.name} ({index.dimension}d, {index.metric})")

except PineconeException as e:
    # 3. Handle specific authentication/network errors safely
    print("❌ Authentication Failed!")
    print(f"🔍 Error details: {str(e)}")

🔄 Testing Pinecone API Key connectivity...
✅ Success! Your Pinecone API key is valid and authenticated.
📁 Available Indexes in your project: 1
   - advanced-rag (1536d, cosine)


# Create an index

In [4]:
index_name = "genai-demo"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# Connect to Index

In [6]:
index = pc.Index(index_name)

# Prepare the documents

In [9]:
# Contains 2 main topics: AI and Cars
documents = [
    "Machine learning is a subset of AI. It is easy to learn",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Rides are good mode of transportation",
    "Cars and automobiles are expensive",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving vehicles are te future"
]

In [10]:
chunks = []
for i, doc in enumerate(documents):
    chunk_item = {
        "_id": str(i),
        "chunk_text": doc
    }
    chunks.append(chunk_item)

print(chunks)

[{'_id': '0', 'chunk_text': 'Machine learning is a subset of AI. It is easy to learn'}, {'_id': '1', 'chunk_text': 'Deep learning uses neural networks'}, {'_id': '2', 'chunk_text': 'Python is used for data science'}, {'_id': '3', 'chunk_text': 'Cars and vehicles are transportation'}, {'_id': '4', 'chunk_text': 'Artificial intelligence is transforming industries'}, {'_id': '5', 'chunk_text': 'Football is a popular sport'}, {'_id': '6', 'chunk_text': 'Data engineering involves pipelines'}, {'_id': '7', 'chunk_text': 'Rides are good mode of transportation'}, {'_id': '8', 'chunk_text': 'Cars and automobiles are expensive'}, {'_id': '9', 'chunk_text': 'Healthcare is improved by AI diagnostics. It is bringing positive change in the world'}, {'_id': '10', 'chunk_text': 'Self-driving vehicles are te future'}]


# Store the documents in vector DB

In [11]:
# Upsert the records into a namespace

index.upsert_records(
    namespace="example-namespace", 
    records=chunks
)

In [13]:
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats_for_index = index.describe_index_stats()
print(stats_for_index)

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 11}},
 'total_vector_count': 11,
 'vector_type': 'dense'}


# Perform Semantic Search

In [14]:
# Define the query
query = "how is AI is changing the world ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

In [15]:
# Returns results sorted by similarity score
print(results)

{'result': {'hits': [{'_id': '9',
                      '_score': 0.4743359684944153,
                      'fields': {'chunk_text': 'Healthcare is improved by AI '
                                               'diagnostics. It is bringing '
                                               'positive change in the world'}},
                     {'_id': '4',
                      '_score': 0.31615668535232544,
                      'fields': {'chunk_text': 'Artificial intelligence is '
                                               'transforming industries'}},
                     {'_id': '0',
                      '_score': 0.164500892162323,
                      'fields': {'chunk_text': 'Machine learning is a subset '
                                               'of AI. It is easy to learn'}},
                     {'_id': '10',
                      '_score': 0.12512488663196564,
                      'fields': {'chunk_text': 'Self-driving vehicles are te '
                          

In [18]:
# Print the results

for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 9     | score: 0.47  | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 4     | score: 0.32  | text: Artificial intelligence is transforming industries
id: 0     | score: 0.16  | text: Machine learning is a subset of AI. It is easy to learn
id: 10    | score: 0.13  | text: Self-driving vehicles are te future               
id: 1     | score: 0.11  | text: Deep learning uses neural networks                
id: 2     | score: 0.1   | text: Python is used for data science                   
id: 6     | score: 0.04  | text: Data engineering involves pipelines               
id: 3     | score: 0.02  | text: Cars and vehicles are transportation              
id: 8     | score: -0.0  | text: Cars and automobiles are expensive                
id: 5     | score: -0.01 | text: Football is a popular sport                       


In [19]:
# Lets use a different query: on  cars
query = "Tell me about rides ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 7     | score: 0.26  | text: Rides are good mode of transportation             
id: 3     | score: 0.08  | text: Cars and vehicles are transportation              
id: 8     | score: 0.05  | text: Cars and automobiles are expensive                
id: 10    | score: 0.05  | text: Self-driving vehicles are te future               
id: 5     | score: 0.04  | text: Football is a popular sport                       
id: 9     | score: -0.01 | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 0     | score: -0.01 | text: Machine learning is a subset of AI. It is easy to learn
id: 4     | score: -0.02 | text: Artificial intelligence is transforming industries
id: 6     | score: -0.03 | text: Data engineering involves pipelines               
id: 1     | score: -0.04 | text: Deep learning uses neural networks                


# Delete the index when done

In [20]:
pc.delete_index(index_name)